In [1]:
import os
from Bio import SeqIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ---------------------------- #
# 📋 Configuration and Setup
# ---------------------------- #

# Define file paths
fasta_file = '/Users/didemdost/Desktop/folds/bacterial_toxins_repseq_0.3.fasta'       # FASTA file path
labels_file = '/Users/didemdost/Desktop/folds/label.csv'          # Labels CSV file path
output_dir = '/Users/didemdost/Desktop/folds/feature_analysis'      # Output directory for plots

# Hydrophobicity scale (Kyte-Doolittle)
HYDROPHOBICITY_SCALE = {
    'A': 1.8,
    'C': 2.5,
    'D': -3.5,
    'E': -3.5,
    'F': 2.8,
    'G': -0.4,
    'H': -3.2,
    'I': 4.5,
    'K': -3.9,
    'L': 3.8,
    'M': 1.9,
    'N': -3.5,
    'P': -1.6,
    'Q': -3.5,
    'R': -4.5,
    'S': -0.8,
    'T': -0.7,
    'V': 4.2,
    'W': -0.9,
    'Y': -1.3
}

# Mapping for 'type' column
TYPE_MAPPING = {
    'TypeIII_controlToxin': 'Type_III',
    'TypeII_PFT_bacteria': 'Type_II',
    'TypeIII': 'Type_III'
}

# ---------------------------- #
# 📚 Define Helper Functions
# ---------------------------- #

def load_fasta(fasta_path):
    """Load sequences from a FASTA file into a DataFrame."""
    print("Parsing FASTA file...")
    records = list(SeqIO.parse(fasta_path, "fasta"))
    data = [{'ID': record.id.split('|')[0].strip(), 'Sequence': str(record.seq).upper()} for record in records]
    print(f"Parsed {len(data)} sequences from FASTA.")
    return pd.DataFrame(data)

def load_labels(labels_path):
    """Load labels from a CSV file into a DataFrame."""
    print("Reading labels CSV file...")
    labels_df = pd.read_csv(labels_path, delimiter=";")
    labels_df['ID'] = labels_df['ID'].astype(str).str.replace('.', '_', regex=False)
    print(f"Loaded {len(labels_df)} label entries.")
    return labels_df[['ID', 'type', 'target']]

def merge_data(sequences_df, labels_df):
    """Merge sequences with labels on 'ID' and map 'type' labels."""
    print("Merging sequences with labels...")
    merged_df = pd.merge(sequences_df, labels_df, on='ID', how='inner')
    merged_df['type'] = merged_df['type'].replace(TYPE_MAPPING)
    merged_df = merged_df.drop_duplicates(subset='ID')
    print(f"Merged data contains {len(merged_df)} entries after removing duplicates.")
    return merged_df

def calculate_hydrophobicity(sequence):
    """Calculate average hydrophobicity of a protein sequence."""
    hydro_values = [HYDROPHOBICITY_SCALE.get(residue, 0) for residue in sequence]
    return np.mean(hydro_values) if hydro_values else 0

def add_features(df):
    """Add hydrophobicity and sequence length features to the DataFrame."""
    print("Calculating hydrophobicity and sequence length...")
    df['Hydrophobicity'] = df['Sequence'].apply(calculate_hydrophobicity)
    df['Sequence_Length'] = df['Sequence'].apply(len)
    print("Feature calculation completed.")
    return df

def create_plot(df, group_col, y_col, plot_title, y_label, filename):
    """Create, display, and save a boxplot with a stripplot overlay."""
    print(f"Creating plot: {plot_title}")
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_col, y=y_col, data=df, palette="Set3", showfliers=False)
    sns.stripplot(x=group_col, y=y_col, data=df, color='0.25', alpha=0.5, jitter=True, size=4)
    plt.title(plot_title, fontsize=14)
    plt.xlabel(group_col.capitalize(), fontsize=12)
    plt.ylabel(y_label, fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path)
    print(f"Saved plot to {save_path}")
    plt.show()  # Display the plot
    plt.close()

def fit_glm(formula, data, family=sm.families.Gaussian()):
    """Fit a Generalized Linear Model and return the results."""
    model = smf.glm(formula=formula, data=data, family=family).fit()
    return model

def summarize_model(model, title, filename):
    """Print and save the summary of the model."""
    print(title)
    print(model.summary())
    summary_path = os.path.join(output_dir, filename)
    with open(summary_path, 'w') as f:
        f.write(title + '\n')
        f.write(model.summary().as_text())
    print(f"Saved model summary to {summary_path}")

# ---------------------------- #
# 🚀 Main Execution Flow
# ---------------------------- #

def main():
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created output directory at {output_dir}.")
    else:
        print(f"Output directory already exists at {output_dir}.")

    # Load data
    print("\nLoading FASTA file...")
    sequences_df = load_fasta(fasta_file)
    print(f"Loaded {len(sequences_df)} sequences.\n")

    print("Loading labels CSV file...")
    labels_df = load_labels(labels_file)
    print(f"Loaded {len(labels_df)} labels.\n")

    # Merge data
    print("Merging sequences with labels...")
    merged_df = merge_data(sequences_df, labels_df)
    print(f"Merged data contains {len(merged_df)} entries after removing duplicates.\n")

    if merged_df.empty:
        print("Error: Merged DataFrame is empty. Please check your FASTA and labels files for matching IDs.")
        return

    # Display first few rows of merged data for verification
    print("First 5 entries of the merged data:")
    print(merged_df.head(), "\n")

    # Add features
    merged_df = add_features(merged_df)

    # Display first few rows after adding features
    print("First 5 entries after adding features:")
    print(merged_df[['ID', 'type', 'target', 'Hydrophobicity', 'Sequence_Length']].head(), "\n")

    # Create plots
    print("Creating plots...\n")

    # Hydrophobicity by type
    create_plot(
        df=merged_df,
        group_col='type',
        y_col='Hydrophobicity',
        plot_title='Hydrophobicity Analysis by Type',
        y_label='Average Hydrophobicity',
        filename='hydrophobicity_by_type.png'
    )

    # Hydrophobicity by target
    create_plot(
        df=merged_df,
        group_col='target',
        y_col='Hydrophobicity',
        plot_title='Hydrophobicity Analysis by Target',
        y_label='Average Hydrophobicity',
        filename='hydrophobicity_by_target.png'
    )

    # Sequence Length by type
    create_plot(
        df=merged_df,
        group_col='type',
        y_col='Sequence_Length',
        plot_title='Sequence Length Analysis by Type',
        y_label='Sequence Length (aa)',
        filename='sequence_length_by_type.png'
    )

    # Sequence Length by target
    create_plot(
        df=merged_df,
        group_col='target',
        y_col='Sequence_Length',
        plot_title='Sequence Length Analysis by Target',
        y_label='Sequence Length (aa)',
        filename='sequence_length_by_target.png'
    )

    print("All plots have been successfully created, displayed, and saved.\n")

    # ---------------------------- #
    # 📊 Bias Analysis with GLMs (Using Sum-to-Zero Coding)
    # ---------------------------- #
    print("Starting bias analysis using Generalized Linear Models (GLMs) with effect coding...\n")

    # Define formulas using sum-to-zero (effect) coding to remove a reference group
    formula_hydro = 'Hydrophobicity ~ C(type, Sum) + C(target, Sum)'
    formula_length = 'Sequence_Length ~ C(type, Sum) + C(target, Sum)'

    # Fit models
    print("Fitting GLM for Hydrophobicity...")
    model_hydro = fit_glm(formula_hydro, merged_df)
    summarize_model(model_hydro, "GLM Summary for Hydrophobicity (Effect Coding)", "glm_summary_hydrophobicity.txt")
    print()

    print("Fitting GLM for Sequence Length...")
    model_length = fit_glm(formula_length, merged_df)
    summarize_model(model_length, "GLM Summary for Sequence Length (Effect Coding)", "glm_summary_sequence_length.txt")
    print()

    # Optional: Visualize residuals to check model assumptions
    def plot_residuals(model, title, filename, residual_type='deviance'):
        """
        Plot residuals of the model.
        
        Parameters:
        - model: Fitted statsmodels GLMResults object
        - title: Title for the plot
        - filename: Filename to save the plot
        - residual_type: Type of residual to plot ('deviance', 'pearson', 'response')
        """
        print(f"Creating residual plot for {title} ({residual_type.capitalize()} Residuals)...")
        if residual_type == 'deviance':
            residuals = model.resid_deviance
        elif residual_type == 'pearson':
            residuals = model.resid_pearson
        elif residual_type == 'response':
            residuals = model.resid_response
        else:
            raise ValueError("Invalid residual_type. Choose from 'deviance', 'pearson', 'response'.")

        fitted = model.fittedvalues
        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=fitted, y=residuals, alpha=0.5)
        plt.axhline(0, color='red', linestyle='--')
        plt.title(f'Residuals vs Fitted for {title}', fontsize=14)
        plt.xlabel('Fitted Values', fontsize=12)
        plt.ylabel(f'{residual_type.capitalize()} Residuals', fontsize=12)
        plt.tight_layout()
        save_path = os.path.join(output_dir, filename)
        plt.savefig(save_path)
        print(f"Saved residual plot to {save_path}")
        plt.show()  # Display the plot
        plt.close()

    # Residual Plots for Hydrophobicity
    plot_residuals(model_hydro, "Hydrophobicity", "residuals_hydrophobicity_deviance.png", residual_type='deviance')
    plot_residuals(model_hydro, "Hydrophobicity", "residuals_hydrophobicity_pearson.png", residual_type='pearson')
    print()

    # Residual Plots for Sequence Length
    plot_residuals(model_length, "Sequence Length", "residuals_sequence_length_deviance.png", residual_type='deviance')
    plot_residuals(model_length, "Sequence Length", "residuals_sequence_length_pearson.png", residual_type='pearson')
    print()

    print("Bias analysis using GLMs with effect coding has been successfully completed. Summaries and residual plots are saved in the output directory.")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'statsmodels'